## MCP: Удалённый вызов инструментов

В этом ноутбуке мы пробуем вызывать удалённые инструменты по протоколу MCP.

Для начала установим необходимые библиотеки:

In [1]:
%pip install openai dotenv

Note: you may need to restart the kernel to use updated packages.


**ВНИМАНИЕ**: После установки библиотек рекомендуется перезапустить Kernel ноутбука.

И ещё полезная функция на будущее:

In [2]:
from IPython.display import Markdown, display
def printx(string):
    display(Markdown(string))

А также сделаем полезные импорты на будущее:

In [3]:
import os
import json

## Авторизация и создание клиента OpenAI

Для работы с языковыми моделями нам понадобится авторизоваться в Yandex Cloud. Для доступа к модели необходимы:
* идентификатор каталога `folder_id`
* API-ключ сервисного аккаунта `api_key`. Сервисный аккаунт должен иметь права на доступ к модели (рекомендуем `ai.editor`).

Мы предполагаем, что соответствующие значения хранятся в переменных окружения. Удобно использовать два подхода:
* Если вы разворачиваете код в Yandex Datasphere - используйте секреты проекта
* Если вы запускаете проект со своего компьютера - разместите значения ключей в файле `.env` (в корневом каталоге репозитория) и запустите следующую ячейку для их загрузки в переменные окружения

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

Создадим клиент OpenAI и убедимся, что он работает:

In [6]:
from openai import OpenAI

model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8/latest"

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id
)

## MCP-сервер на виртуальной машине

Простейший способ реализовать свой MCP-сервер - это использовать библиотеку [FastMCP](https://gofastmcp.com/). Мы реализуем функции сервера в виде обычных функций Python, и декорируем их с помощью `@mcp.tool`, примерно следующим образом:

```python
@mcp.tool(description="Сложить два числа")
def add_numbers(a: int, b: int) -> int:
    return a+b
```
При этом типизация всех аргументов и описание функции `description` позволяют сформировать правильную JSON-схему для LLM. Дополнительно помогает добавить функции подробное docstring-описание, со словесным описанием всех аргументов и результата функции.

Предположим, что мы хотим организовать MCP-сервер для хранения персональных заметок. В нем заметки будут распределены по записным книжкам, и функция добавления заметок будет иметь такое описание:

```python
@mcp.tool(description="Добавить заметку в блокнот")
def add_note(
    title: str, body: str,
    notebook: str | None = None,
) -> dict:
    """Создаёт новую заметку и сохраняет её в блокнот.

    Args:
        title: Заголовок заметки. Не может быть пустым.
        body: Текст заметки. Не может быть пустым.
        notebook: Имя блокнота. Если не указано, используется "scrapbook".

    Returns:
        Словарь с полями ``id``, ``notebook``, ``created_at``, ``title``, ``body``.
    """
    ... 
```

Также имеет смысл реализовать функцию нахождения списка заметок в заданной записной книжке, перемещения заметки в другую книжку и т.д.

Простейшая реализация такого MCP-сервера с хранением заметок в памяти приведена в файле [notes.py](../mcp-servers/notes.py).

Для запуска MCP-сервера в облаке Yandex Cloud можно воспользоваться виртуальной машиной. Понадобится установить на неё реализацию python и необходимые библиотеки. После этого, запускаем сервер командой
```sh
python notes.py
```
или
```sh
fastmcp run notes.py -t sse --host 0.0.0.0
```

Для вызова сервера из Responses API опишем инструмент вот таким словарём:

In [18]:
notes_tool = {
    "type": "mcp",
    "server_label": "PersonalNotes",
    "server_url": "http://cathy.ycloud.eazify.net:8000/sse", # это адрес виртуальной машины
    "require_approval": "never",
}

И укажем инструмент в запросе к Responses API:

In [19]:
res = client.responses.create(
    model=model,
    tools=[notes_tool],
    input="Добавь в мой дневник заметку о том, что я сегодня купил хлеб и молоко",
)
printx(res.output_text)

Заметка успешно добавлена в блокнот "дневник":

**Заголовок:** Покупки  
**Текст:** Сегодня купил хлеб и молоко.  
**Дата создания:** 18 февраля 2026 года

In [20]:
for x in res.output:
    if x.type=='mcp_call':
        print(f" + Вызов {x.name}{x.arguments}")
        print(f"   Результат: {x.output}")

 + Вызов add_note{"title": "Покупки", "body": "Сегодня купил хлеб и молоко.", "notebook": "дневник"}
   Результат: {"id":1,"notebook":"дневник","created_at":"2026-02-18T09:04:51Z","title":"Покупки","body":"Сегодня купил хлеб и молоко."}


Посмотрим, какие заметки есть в блокноте:

In [ ]:
res = client.responses.create(
    model=model,
    tools=[notes_tool],
    input="Какие у меня есть записи?",
)
printx(res.output_text)

У вас есть одна запись:

- **Заголовок:** Покупки  
  **Блокнот:** дневник  
  **Дата создания:** 17 февраля 2026  
  **Текст:** Сегодня купил хлеб и молоко.

Это последняя запись из вашего блокнота. Если нужно, могу показать больше записей или помочь с поиском.

## MCP Hub

В AI Studio существует специальный инструмент для подключения различных сервисов к агентам по протоколу MCP - [MCP Hub](https://yandex.cloud/ru/docs/ai-studio/concepts/mcp-hub). Он позволяет интегрировать не только существующие внешние MCP-сервера, но и сервисы, доступные через обычные REST API.

Например, можно обернуть в MCP-сервер открытый сервис [OpenWeatherMap](https://openweathermap.org) для запроса погоды. Он позволяет получить погоду по географическим координатам, или по названию города. Достаточно послать GET-запрос следующего вида: `https://api.openweathermap.org/data/2.5/weather?q={city name}&appid={API key}`. Дополнительно можно также передать `untis=metric`, чтобы получить ответ в градусах Цельсия. Ключ API_key можно получить, зарегистрировавшись на сайте.

Как создать MCP-сервер из этого запроса - смотрите [в видео](https://youtu.be/gshhvxjNLzU). После того, как мы получили адрес сервера в MCP Hub, можем вызывать его обычным способом:

In [13]:
weather_tool = {
    "type": "mcp",
    "server_label": "weather",
    "server_url": "https://db8ubou2m0vmu17biigk.58zke0qh.mcpgw.serverless.yandexcloud.net/sse",
    "require_approval": "never",
}

res = client.responses.create(
    model=model,
    tools=[weather_tool],
    input="Какая погода в Москве?",
)
printx(res.output_text)

В Москве сейчас:

- **Температура**: -11.43 °C, ощущается как -17.91 °C  
- **Погода**: Малооблачно (few clouds)  
- **Влажность**: 85%  
- **Давление**: 1015 гПа  
- **Скорость ветра**: 3.59 м/с, порывы до 5.77 м/с  
- **Видимость**: 10 км  
- **Облачность**: 22%  

Рассвет был в 08:10, закат в 17:56 (по местному времени).

Посмотрим на структуру ответа с учётом вызова MCP-сервера:

In [16]:
for x in res.output:
    if x.type=='mcp_call':
        print(f" + Вызов {x.name}{x.arguments}")
        print(f"   Результат: {x.output}")

 + Вызов getweather{"city": "Moscow"}
   Результат: {"coord":{"lon":37.6156,"lat":55.7522},"weather":[{"id":801,"main":"Clouds","description":"few clouds","icon":"02d"}],"base":"stations","main":{"temp":-11.43,"feels_like":-17.91,"temp_min":-11.46,"temp_max":-10.71,"pressure":1015,"humidity":85,"sea_level":1015,"grnd_level":995},"visibility":10000,"wind":{"speed":3.59,"deg":352,"gust":5.77},"clouds":{"all":22},"dt":1771326993,"sys":{"type":2,"id":2000314,"country":"RU","sunrise":1771303826,"sunset":1771339018},"timezone":10800,"id":524901,"name":"Moscow","cod":200}


## MCP-сервер из облачной функции

В предыдущем примере, мы отобразили параметры MCP-сервера (город **city**) напрямую на параметры REST-запроса. Однако иногда бывает так, что нужно совершить чуть более сложное преобразование данных. Например, сервис OpenWeatherMap рекомендует отдельно сначала вызывать сервис геолокации, и затем уже передавать в сервис определения погоды конкретные координаты широты и долготы.

Такие небольшие фрагменты кода, выполняемые в облаке, удобно реализовывать в виде **облачных функций** (Cloud Functions) в парадигме **бессерверных вычислений**. Создать облачную функцию можно из облачной консоли, вписав код прямо в редактор:

```python
import os
import requests

def handler(event, context):
    city = event['city']
    api_key = os.environ['api_key'] 
    geo = requests.get(f"http://api.openweathermap.org/geo/1.0/direct?q={city}&limit=1&appid={api_key}").json()
    lat = geo[0]['lat']
    lon = geo[0]['lon']
    res = requests.get(f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={api_key}")        
    return {
        'statusCode': 200,
        'body': res.json()
    }
```

Процесс создания такой облачной функции и MCP-сервера на её основе показан [в этом видео](https://www.youtube.com/watch?v=jl-V79SCWeo).

## Выводы

К вызову LLM достаточно легко добавить инструмент поиска, и все соответствующие сложности по вызову инструмента и обработке результата Yandex Cloud берёт на себя. Однако, следует помнить, что инструмент поиска [тарифицируется](https://yandex.cloud/ru/docs/ai-studio/pricing) отдельно, и это увеличивает стоимость запросов. Также, LLM сама решает, когда следует вызывать инструмент поиска, поэтому стоит указать какие-то правила вызова поиска в системном промпте и/или в самом запросе. 